In [3]:
import re
import os
import copy
import json
import random
import pickle
import logging
from pathlib import Path
from datetime import datetime
import networkx as nx
import matplotlib.pyplot as plt

In [4]:
ruta_archivo_norequ = r"C:\Users\alhel\Desktop\TESIS CARP 2025\Instancias\eglese\egl-e1-A.dat" #this is the most general case

# Read instance .dat file based on format described in https://www.uv.es/~belengue/carp/READ_ME

In [5]:
ruta_archivo_norequ

'C:\\Users\\alhel\\Desktop\\TESIS CARP 2025\\Instancias\\eglese\\egl-e1-A.dat'

In [6]:
import re

def leer_carplib_dat(filepath):
    parsed_data = {}
    current_list_key = None
    contador_tarea = 1
    
    with open(filepath, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith(('=', '-', '|')):
                continue
            
            if ':' in line:
                key, val = line.split(':', 1)
                key = key.strip()
                val = val.strip()
                
                if key.startswith('LISTA_ARISTAS'):
                    current_list_key = key
                    parsed_data[key] = []
                    contador_tarea = 1
                else:
                    current_list_key = None
                    if val.isdigit() or (val.startswith('-') and val[1:].isdigit()):
                        parsed_data[key] = int(val)
                    else:
                        parsed_data[key] = val
                        
            elif current_list_key:
                numeros = [int(x) for x in re.findall(r'-?\d+', line)]
                if len(numeros) >= 3:
                    # Asignamos el prefijo dependiendo de la lista que se esté leyendo
                    if "NOREQ" in current_list_key:
                        prefijo = "TNR"
                    else:
                        prefijo = "TR"
                        
                    parsed_data[current_list_key].append({
                        'tarea': f"{prefijo}{contador_tarea}",
                        'nodos': (numeros[0], numeros[1]),
                        'costo': numeros[2],
                        'demanda': numeros[3] if len(numeros) >= 4 else 0
                    })
                    contador_tarea += 1

    nombre_objeto = 'loaded_dict'  
    return parsed_data, nombre_objeto

In [7]:
d_noreq, d_noreq_sname = leer_carplib_dat(ruta_archivo_norequ)
d_noreq, d_noreq_sname

({'NOMBRE': 'egl-e1-A',
  'COMENTARIO': '3625. (cota superior)',
  'VERTICES': 77,
  'ARISTAS_REQ': 51,
  'ARISTAS_NOREQ': 47,
  'VEHICULOS': 5,
  'CAPACIDAD': 305,
  'TIPO_COSTES_ARISTAS': 'EXPLICITOS',
  'COSTE_TOTAL_REQ': 1468,
  'LISTA_ARISTAS_REQ': [{'tarea': 'TR1',
    'nodos': (1, 2),
    'costo': 32,
    'demanda': 32},
   {'tarea': 'TR2', 'nodos': (2, 3), 'costo': 14, 'demanda': 14},
   {'tarea': 'TR3', 'nodos': (2, 4), 'costo': 17, 'demanda': 17},
   {'tarea': 'TR4', 'nodos': (4, 5), 'costo': 56, 'demanda': 56},
   {'tarea': 'TR5', 'nodos': (9, 10), 'costo': 20, 'demanda': 20},
   {'tarea': 'TR6', 'nodos': (11, 12), 'costo': 32, 'demanda': 32},
   {'tarea': 'TR7', 'nodos': (12, 16), 'costo': 29, 'demanda': 29},
   {'tarea': 'TR8', 'nodos': (13, 16), 'costo': 13, 'demanda': 13},
   {'tarea': 'TR9', 'nodos': (13, 14), 'costo': 7, 'demanda': 7},
   {'tarea': 'TR10', 'nodos': (15, 17), 'costo': 26, 'demanda': 26},
   {'tarea': 'TR11', 'nodos': (15, 18), 'costo': 38, 'demanda': 38

In [8]:
d_noreq.values()

dict_values(['egl-e1-A', '3625. (cota superior)', 77, 51, 47, 5, 305, 'EXPLICITOS', 1468, [{'tarea': 'TR1', 'nodos': (1, 2), 'costo': 32, 'demanda': 32}, {'tarea': 'TR2', 'nodos': (2, 3), 'costo': 14, 'demanda': 14}, {'tarea': 'TR3', 'nodos': (2, 4), 'costo': 17, 'demanda': 17}, {'tarea': 'TR4', 'nodos': (4, 5), 'costo': 56, 'demanda': 56}, {'tarea': 'TR5', 'nodos': (9, 10), 'costo': 20, 'demanda': 20}, {'tarea': 'TR6', 'nodos': (11, 12), 'costo': 32, 'demanda': 32}, {'tarea': 'TR7', 'nodos': (12, 16), 'costo': 29, 'demanda': 29}, {'tarea': 'TR8', 'nodos': (13, 16), 'costo': 13, 'demanda': 13}, {'tarea': 'TR9', 'nodos': (13, 14), 'costo': 7, 'demanda': 7}, {'tarea': 'TR10', 'nodos': (15, 17), 'costo': 26, 'demanda': 26}, {'tarea': 'TR11', 'nodos': (15, 18), 'costo': 38, 'demanda': 38}, {'tarea': 'TR12', 'nodos': (18, 19), 'costo': 41, 'demanda': 41}, {'tarea': 'TR13', 'nodos': (19, 20), 'costo': 32, 'demanda': 32}, {'tarea': 'TR14', 'nodos': (19, 21), 'costo': 38, 'demanda': 38}, {'tar

## CHECKING THAT THE NUMBER OF LINES IS THE SAME AS THE NUMBER OF ITEMS IN DICT

In [9]:
def validate_instance(created_dict, dat_file_path): ### las no requeridas deben tener otro id, no Ti
    """
    Valida la integridad de la instancia comparando metadatos, 
    la longitud de las listas y el total de líneas útiles en el archivo original.
    """
    
    # 1. Extraemos los valores esperados (lo que dice el encabezado)
    n_req_esperadas = created_dict.get("ARISTAS_REQ", 0)
    n_noreq_esperadas = created_dict.get("ARISTAS_NOREQ", 0)
    
    # 2. Contamos cuánto se leyó realmente en las listas
    lista_req = created_dict.get("LISTA_ARISTAS_REQ", [])
    lista_noreq = created_dict.get("LISTA_ARISTAS_NOREQ", [])
    n_req_reales = len(lista_req)
    n_noreq_reales = len(lista_noreq)
    
    # 3. Validamos integridad de nodos (el nodo más alto no debe superar VERTICES)
    todas_las_aristas = lista_req + lista_noreq
    
    nodos_encontrados = []
    for item in todas_las_aristas:
        # Corregido: usamos 'nodos' en lugar de 'arco' basado en nuestra función anterior
        nodos_encontrados.extend(item['nodos'])
    
    max_nodo = max(nodos_encontrados) if nodos_encontrados else 0
    total_vertices = created_dict.get("VERTICES", 0)

    # 4. NUEVA LÓGICA: Contar líneas útiles en el archivo .dat
    lineas_utiles_archivo = 0
    with open(dat_file_path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            # Aplicamos el mismo filtro que en la función de lectura
            if not line or line.startswith('=') or line.startswith('-') or line.startswith('|'):
                continue
            lineas_utiles_archivo += 1

    # El total de elementos capturados es: llaves principales + items en las listas
    elementos_en_dict = len(created_dict) + n_req_reales + n_noreq_reales

    # 5. Verificación de condiciones
    check_req = (n_req_esperadas == n_req_reales)
    check_noreq = (n_noreq_esperadas == n_noreq_reales)
    check_nodes = (max_nodo <= total_vertices)
    check_lineas = (lineas_utiles_archivo == elementos_en_dict)

    is_valid = check_req and check_noreq and check_nodes and check_lineas

    # 6. Resultado detallado
    validation_results = {
        "is_valid": is_valid,
        "status": "VALIDACIÓN EXITOSA" if is_valid else "ERROR DE INTEGRIDAD",
        "detalles": {
            "aristas_req": f"{n_req_reales}/{n_req_esperadas}",
            "aristas_noreq": f"{n_noreq_reales}/{n_noreq_esperadas}",
            "nodos_ok": f"Max Nodo {max_nodo} <= {total_vertices}",
            "lineas_vs_dict": f"Archivo: {lineas_utiles_archivo} == Dict: {elementos_en_dict}"
        }
    }

    nombre_objeto = 'validation_dict'
    return validation_results, nombre_objeto

In [10]:
reading_validation_dict_noreq, val_d_name = validate_instance(d_noreq, ruta_archivo_norequ)
reading_validation_dict_noreq, val_d_name

({'is_valid': True,
  'status': 'VALIDACIÓN EXITOSA',
  'detalles': {'aristas_req': '51/51',
   'aristas_noreq': '47/47',
   'nodos_ok': 'Max Nodo 77 <= 77',
   'lineas_vs_dict': 'Archivo: 110 == Dict: 110'}},
 'validation_dict')

In [11]:
val_d_name

'validation_dict'

# GESTIÓN DE EJECUCIÓN Y GUARDADO

In [12]:
def iniciar_ejecucion(metadata_dict, base_dir="instance_runs"):
    project_name = metadata_dict.get("NOMBRE", "Instancia")
    run_id = datetime.now().strftime("%H%M%S")
    date_str = datetime.now().strftime("%Y-%m-%d")

    try:
        directorio_base = Path(__file__).resolve().parent
    except NameError:
        directorio_base = Path.cwd()
        
    run_path = directorio_base / base_dir / f"{date_str}_{project_name}_ID-{run_id}"
    run_path.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(f"{project_name}_{run_id}")
    logger.setLevel(logging.INFO)
    
    if not logger.handlers:
        file_handler = logging.FileHandler(run_path / "execution.log")
        file_handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
        logger.addHandler(file_handler)

    logger.info("Run initialized.")
    return run_path, logger

In [13]:
tst_path, tst_logger = iniciar_ejecucion(d_noreq, base_dir="instance_runs_test_debug_20260503")

In [221]:
tst_path

WindowsPath('C:/Users/alhel/Desktop/TESIS CARP 2025/Metaheuristicas/TestingClass/instance_runs_test_debug_20260503/2026-03-10_egl-e1-A_ID-132343')

In [15]:
import os
import json
import pickle

def guardar_objeto_automatico(carpeta, nombre_instancia, objeto, tipo_objeto):
    """
    Guarda un objeto en disco infiriendo su formato ideal.
    El archivo resultante tendrá el formato: carpeta/nombre_instancia_tipo_objeto.ext
    """
    # Construimos el nombre base estandarizado
    nombre_base = f"{nombre_instancia}_{tipo_objeto}"
    
    # 1. Gráficas o figuras (Matplotlib/Plotly compatibles)
    if hasattr(objeto, 'savefig'):
        ruta = os.path.join(carpeta, f"{nombre_base}.jpg")
        objeto.savefig(ruta, format='jpg', dpi=300, bbox_inches='tight')
        return ruta
        
    # 2. Cadenas de texto plano (Logs, reportes)
    elif isinstance(objeto, str):
        ruta = os.path.join(carpeta, f"{nombre_base}.txt")
        with open(ruta, 'w', encoding='utf-8') as f:
            f.write(objeto)
        return ruta
        
    # 3. Diccionarios o Listas (Intentamos guardar como JSON para legibilidad)
    elif isinstance(objeto, (dict, list)):
        ruta_json = os.path.join(carpeta, f"{nombre_base}.json")
        try:
            with open(ruta_json, 'w', encoding='utf-8') as f:
                json.dump(objeto, f, indent=4)
            return ruta_json
        except TypeError:
            # Si el diccionario tiene objetos no serializables en JSON (como tuplas como llaves), 
            # fallará silenciosamente y pasará al guardado en Pickle.
            pass 
            
    # 4. Fallback de seguridad (Pickle binario para cualquier objeto complejo)
    ruta_pkl = os.path.join(carpeta, f"{nombre_base}.pkl")
    with open(ruta_pkl, 'wb') as f:
        pickle.dump(objeto, f)
        
    return ruta_pkl

In [16]:
 guardar_objeto_automatico(tst_path, d_noreq['NOMBRE'],d_noreq, d_noreq_sname)

'C:\\Users\\alhel\\Desktop\\TESIS CARP 2025\\Metaheuristicas\\TestingClass\\instance_runs_test_debug_20260503\\2026-03-10_egl-e1-A_ID-132343\\egl-e1-A_loaded_dict.json'

In [17]:
 guardar_objeto_automatico(tst_path, d_noreq['NOMBRE'],reading_validation_dict_noreq, val_d_name)

'C:\\Users\\alhel\\Desktop\\TESIS CARP 2025\\Metaheuristicas\\TestingClass\\instance_runs_test_debug_20260503\\2026-03-10_egl-e1-A_ID-132343\\egl-e1-A_validation_dict.json'

In [8]:
def finalizar_ejecucion(run_path, logger, success=True, reason=""):
    status_str = "SUCCESS" if success else "FAILED"
    (run_path / f"STATUS_{status_str}").touch()
    
    mensaje = f"Run finished with status: {status_str}"
    if reason: 
        mensaje += f" | Reason: {reason}"
        
    logger.info(mensaje)

# Draw the graph based on the arcs and nodes extracted in the previous step

In [222]:
def generar_grafo_limpio(data, carpeta_salida=None, figsize=(35,30), k_layout=15):
    G = nx.Graph()
    for i in range(1, data.get('VERTICES', 0) + 1):
        G.add_node(i)
        
    for item in data.get('LISTA_ARISTAS_REQ', []):
        u, v = item['nodos']
        G.add_edge(u, v, cost=item['costo'], demand=item['demanda'], tipo='req')

    for item in data.get('LISTA_ARISTAS_NOREQ', []):
        u, v = item['nodos']
        G.add_edge(u, v, cost=item['costo'], demand=item.get('demanda', 0), tipo='noreq')

    pos = nx.spring_layout(G, k=k_layout, iterations=200, seed=42)
    plt.figure(figsize=figsize) 
    
    deposito = data.get('DEPOSITO', 1)
    node_colors = ['#ff4d4d' if node == deposito else '#74b9ff' for node in G.nodes()]
    
    nx.draw_networkx_nodes(G, pos, node_size=900, node_color=node_colors, edgecolors='#74b9ff', linewidths=1)
    nx.draw_networkx_labels(G, pos, font_size=15, font_weight='bold')
    
    arcos_req = [(u, v) for u, v, d in G.edges(data=True) if d['tipo'] == 'req']
    arcos_noreq = [(u, v) for u, v, d in G.edges(data=True) if d['tipo'] == 'noreq']
    
    nx.draw_networkx_edges(G, pos, edgelist=arcos_req, width=2.0, edge_color='black')
    nx.draw_networkx_edges(G, pos, edgelist=arcos_noreq, width=1.5, edge_color='gray', style='dashed')
    
    edge_labels = {}
    for u, v, d in G.edges(data=True):
        if d['tipo'] == 'req':
            edge_labels[(u, v)] = f"C:{d['cost']}\nD:{d['demand']}"
        else:
            edge_labels[(u, v)] = f"C:{d['cost']}"
            
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=15, font_color='#2d3436', label_pos=0.5, bbox=dict(alpha=0))
    
    texto_resumen = (
        f"RESUMEN DE LA INSTANCIA\n"
        f"--------------------------------------\n"
        f"Nombre: {data.get('NOMBRE', 'N/A')}\n"
        f"Vértices: {data.get('VERTICES', 0)}\n"
        f"Vehículos: {data.get('VEHICULOS', 0)}\n"
        f"Capacidad: {data.get('CAPACIDAD', 0)}\n"
        f"Aristas Req: {data.get('ARISTAS_REQ', 0)}\n"
        f"Aristas No Req: {data.get('ARISTAS_NOREQ', 0)}"
    )
    
    plt.text(0.02, 0.98, texto_resumen, transform=plt.gca().transAxes, fontsize=20, 
             verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', facecolor='#f1f2f6', alpha=0.9))

    nombre_instancia = data.get('NOMBRE', 'instancia')
    plt.title(f"Grafo de Instancia: {nombre_instancia}\n"
              f"(Rojo = Depósito | Negro Sólido = Req | Gris Punteado = No Req)", 
              fontsize=25, pad=20)
    plt.axis('off')
    plt.tight_layout()

    # NUEVO: Guardar la imagen de Matplotlib directamente aquí
    # (Asumiendo que le pasas 'carpeta_salida' a la función)
    if 'carpeta_salida' in locals():
        ruta_imagen = os.path.join(carpeta_salida, f"{nombre_instancia}_mapa_estatico.png")
        plt.savefig(ruta_imagen, format='png', dpi=300, bbox_inches='tight')
  #  plt.show()
    plt.close() # Cierra la figura visual
    
    #object_name = "grafo_matematico"
    return G

In [223]:
G_test = generar_grafo_limpio(d_noreq, carpeta_salida = tst_path, figsize=(35,30), k_layout=15)
G_test

# Generar la distancia mas corta (costo) entre cada par de nodos

In [50]:
import networkx as nx
import matplotlib.pyplot as plt

In [51]:
import networkx as nx

def calcular_matriz_distancias(G, algoritmo='dijkstra', carpeta_salida=None, nombre_instancia="instancia"):
    algoritmo = algoritmo.strip().lower()
    nodos = list(G.nodes())
    matriz_distancias = {}
    
    if algoritmo == 'dijkstra':
        dijkstra_raw = dict(nx.all_pairs_dijkstra_path_length(G, weight='cost'))
        for u in nodos:
            matriz_distancias[u] = {}
            for v in nodos:
                matriz_distancias[u][v] = dijkstra_raw.get(u, {}).get(v, float('inf'))
                
    elif algoritmo in ['floyd-warshall', 'floyd_warshall', 'floyd']:
        fw_raw = nx.floyd_warshall(G, weight='cost')
        for u in nodos:
            matriz_distancias[u] = dict(fw_raw[u])
            
    # Añadido para hacer match con el selector de la interfaz gráfica (app.py)
    elif algoritmo in ['bellman-ford', 'bellman_ford', 'bellman']:
        bf_raw = dict(nx.all_pairs_bellman_ford_path_length(G, weight='cost'))
        for u in nodos:
            matriz_distancias[u] = {}
            for v in nodos:
                matriz_distancias[u][v] = bf_raw.get(u, {}).get(v, float('inf'))
                
    else:
        raise ValueError(f"Algoritmo '{algoritmo}' no reconocido.")
        
    # NUEVO: Guardado Automático en disco
    if carpeta_salida:
        # El nombre quedará como: nombre_instancia_matriz_distancias_dijkstra.json
        tipo_objeto = f"matriz_distancias_{algoritmo}"
        # Se asume que guardar_objeto_automatico está definida en este mismo archivo
        guardar_objeto_automatico(carpeta_salida, nombre_instancia,  matriz_distancias, tipo_objeto)
        
    return matriz_distancias

In [52]:
matriz_distancias_t = calcular_matriz_distancias(G_test, algoritmo='dijkstra', carpeta_salida=tst_path, nombre_instancia = d_noreq['NOMBRE'])
matriz_distancias_t

{1: {1: 0,
  2: 32,
  3: 46,
  4: 49,
  5: 105,
  6: 113,
  7: 111,
  8: 129,
  9: 155,
  10: 175,
  11: 187,
  12: 219,
  13: 261,
  14: 268,
  15: 274,
  16: 248,
  17: 300,
  18: 255,
  19: 256,
  20: 234,
  21: 230,
  22: 247,
  23: 288,
  24: 251,
  25: 278,
  26: 295,
  27: 310,
  28: 313,
  29: 305,
  30: 342,
  31: 328,
  32: 367,
  33: 376,
  34: 410,
  35: 281,
  36: 403,
  37: 301,
  38: 309,
  39: 280,
  40: 275,
  41: 266,
  42: 248,
  43: 241,
  44: 163,
  45: 175,
  46: 176,
  47: 209,
  48: 219,
  49: 218,
  50: 226,
  51: 228,
  52: 228,
  53: 232,
  54: 232,
  55: 260,
  56: 254,
  57: 234,
  58: 156,
  59: 135,
  60: 189,
  61: 228,
  62: 212,
  63: 242,
  64: 251,
  65: 263,
  66: 228,
  67: 218,
  68: 273,
  69: 124,
  70: 409,
  71: 353,
  72: 347,
  73: 352,
  74: 377,
  75: 271,
  76: 226,
  77: 261},
 2: {1: 32,
  2: 0,
  3: 14,
  4: 17,
  5: 73,
  6: 81,
  7: 79,
  8: 97,
  9: 123,
  10: 143,
  11: 155,
  12: 187,
  13: 229,
  14: 236,
  15: 242,
  16: 216,
  

# Factibilidad

In [198]:
import random

def es_ruta_factible(ruta, data, matriz_distancias):
    """Verifica si una sola ruta cumple con capacidad y conectividad."""
    if not ruta: return True
    
    capacidad_max = data.get('CAPACIDAD', 0)
    deposito = data.get('DEPOSITO', 1)
    info_tareas = {t['tarea']: t for t in data.get('LISTA_ARISTAS_REQ', [])}
    
    demanda_total = 0
    nodos_anteriores = (deposito, deposito)
    
    for id_tarea in ruta:
        tarea = info_tareas.get(id_tarea)
        if not tarea: return False
        
        u_act, v_act = tarea['nodos']
        demanda_total += tarea['demanda']
        
        # 1. Chequeo de Capacidad
        if demanda_total > capacidad_max: 
            return False
            
        # 2. Chequeo de Conectividad (¿Podemos llegar desde la tarea anterior a esta?)
        u_ant, v_ant = nodos_anteriores
        hay_camino = (
            matriz_distancias[u_ant][u_act] != float('inf') or 
            matriz_distancias[u_ant][v_act] != float('inf') or
            matriz_distancias[v_ant][u_act] != float('inf') or 
            matriz_distancias[v_ant][v_act] != float('inf')
        )
        
        if not hay_camino: return False
        nodos_anteriores = (u_act, v_act)
        
    # 3. Retorno al depósito (¿Podemos volver a casa desde la última tarea?)
    u_ant, v_ant = nodos_anteriores
    if matriz_distancias[u_ant][deposito] == float('inf') and matriz_distancias[v_ant][deposito] == float('inf'):
        return False
        
    return True

In [199]:
def es_solucion_factible(solucion, data, matriz_distancias):
    """Verifica si toda la solución (flota de vehículos) es válida."""
    num_vehiculos = data.get('VEHICULOS', 0)
    
    # 1. Checar que no se utilicen más camiones de los permitidos
    rutas_activas = [r for r in solucion if r]
    if len(rutas_activas) > num_vehiculos:
        return False
        
    # 2. Validar cada ruta individualmente
    for ruta in solucion:
        if not es_ruta_factible(ruta, data, matriz_distancias):
            return False
            
    return True

# SOLUCION ALEATORIA

In [56]:
def generar_solucion_inicial_aleatoria(data, matriz_distancias, max_intentos=10000, carpeta_salida=None, nombre_instancia="instancia"):
    """Construye aleatoriamente una solución y delega la validación."""
    num_vehiculos = data.get('VEHICULOS', 0)
    capacidad_max = data.get('CAPACIDAD', 0)
    tareas_requeridas = data.get('LISTA_ARISTAS_REQ', [])
    
    # Chequeo rápido de viabilidad base
    for t in tareas_requeridas:
        if t['demanda'] > capacidad_max: 
            # Siguiendo tu instrucción de formato para tareas (mostrar el arco)
            arco = t.get('nodos', 'Desconocido')
            raise ValueError(f"Instancia inviable: La tarea {t['tarea']} (Arco: {arco}) excede la capacidad máxima.")

    intentos = 0
    
    while intentos < max_intentos:
        intentos += 1
        tareas_mezcladas = tareas_requeridas.copy()
        random.shuffle(tareas_mezcladas)
        
        solucion = [[] for _ in range(num_vehiculos)]
        v_idx = 0
        demanda_actual = 0
        
        # 1. Construcción (Agrupamiento voraz guiado por capacidad)
        for tarea in tareas_mezcladas:
            if demanda_actual + tarea['demanda'] <= capacidad_max:
                solucion[v_idx].append(tarea['tarea'])
                demanda_actual += tarea['demanda']
            else:
                v_idx += 1
                if v_idx >= num_vehiculos:
                    # Nos quedamos sin camiones, metemos la tarea al último para forzar 
                    # que el validador lo rechace y volvamos a intentarlo.
                    solucion[-1].append(tarea['tarea'])
                    break 
                    
                solucion[v_idx].append(tarea['tarea'])
                demanda_actual = tarea['demanda']
                
        # 2. Validación (Delegada a las funciones expertas)
        if es_solucion_factible(solucion, data, matriz_distancias):
            # NUEVO: Guardado automático de la primera solución aleatoria válida
            if carpeta_salida:
                guardar_objeto_automatico(carpeta_salida, nombre_instancia, solucion, "initial_random_solution")
            return solucion
            
    raise RuntimeError(f"No se halló solución factible tras {max_intentos} intentos.")

In [57]:
random_sol_t = generar_solucion_inicial_aleatoria(d_noreq, matriz_distancias_t, max_intentos=10000,
                                                  carpeta_salida=tst_path, nombre_instancia=d_noreq['NOMBRE'])
random_sol_t

[['TR10', 'TR21', 'TR41', 'TR26', 'TR37', 'TR19', 'TR34'],
 ['TR1', 'TR51', 'TR17', 'TR23', 'TR4', 'TR25', 'TR7', 'TR36'],
 ['TR38',
  'TR11',
  'TR9',
  'TR47',
  'TR50',
  'TR28',
  'TR31',
  'TR14',
  'TR6',
  'TR30',
  'TR24',
  'TR29',
  'TR45'],
 ['TR39',
  'TR15',
  'TR33',
  'TR49',
  'TR44',
  'TR13',
  'TR12',
  'TR20',
  'TR16',
  'TR22',
  'TR8',
  'TR43'],
 ['TR2',
  'TR27',
  'TR35',
  'TR5',
  'TR18',
  'TR46',
  'TR48',
  'TR42',
  'TR3',
  'TR32',
  'TR40']]

In [58]:
es_solucion_factible(random_sol_t, d_noreq, matriz_distancias_t)

True

In [68]:
def detalle_sol(solucion, data, G, carpeta_salida=None, nombre_instancia="instancia" ):
    
    deposito = data.get('DEPOSITO', 1)
    capacidad_max = data.get('CAPACIDAD', 0)
    info_tareas = {t['tarea']: {'u': t['nodos'][0], 'v': t['nodos'][1], 'costo': t['costo'], 'demanda': t['demanda']}
                   for t in data.get('LISTA_ARISTAS_REQ', [])}
        
    costos_rutas = []
    costo_total_solucion = 0
    reporte = [] # Guardaremos el reporte en una lista para retornarlo como string
    
    reporte.append("="*80)
    reporte.append("EVALUACIÓN COMPACTA DE RUTAS")
    reporte.append("="*80)
    
    for i, ruta in enumerate(solucion):
        reporte.append(f"RUTA {i + 1} {ruta}")
        if not ruta:
            reporte.append(f"  -> Vehículo vacío | Costo Total: 0 | Demanda: 0 / {capacidad_max}\n")
            costos_rutas.append(0)
            continue
            
        costo_vehiculo, demanda_vehiculo, nodo_actual = 0, 0, deposito
        for id_tarea in ruta:
            tarea = info_tareas[id_tarea]
            u, v, costo_serv, dem_serv = tarea['u'], tarea['v'], tarea['costo'], tarea['demanda']
            
            if nodo_actual != u:
                camino_dh = nx.shortest_path(G, source=nodo_actual, target=u, weight='cost')
                costo_dh = nx.shortest_path_length(G, source=nodo_actual, target=u, weight='cost')
                str_dh = " -> ".join(map(str, camino_dh))
            else:
                costo_dh, str_dh = 0, f"Ninguno (ya en {u})"
                
            costo_total_paso = costo_dh + costo_serv
            costo_vehiculo += costo_total_paso
            demanda_vehiculo += dem_serv
            
            reporte.append(f"  -> {id_tarea} ({u},{v}) -> DH: [{str_dh}] | Demanda: {dem_serv} | Costo (DH + Serv): {costo_dh} + {costo_serv} = {costo_total_paso}")
            nodo_actual = v
            
        if nodo_actual != deposito:
            camino_ret = nx.shortest_path(G, source=nodo_actual, target=deposito, weight='cost')
            costo_ret = nx.shortest_path_length(G, source=nodo_actual, target=deposito, weight='cost')
            str_ret = " -> ".join(map(str, camino_ret))
        else:
            costo_ret, str_ret = 0, f"Ninguno (ya en {deposito})"
            
        costo_vehiculo += costo_ret
        reporte.append(f"  -> REGRESO A DEPÓSITO ({deposito}) -> DH: [{str_ret}] | Costo Regreso: {costo_ret}")
        
        estado_cap = "OK" if demanda_vehiculo <= capacidad_max else "EXCEDIDA"
        reporte.append(f"  => TOTAL RUTA {i + 1}: Costo Total = {costo_vehiculo} | Demanda Total = {demanda_vehiculo} / {capacidad_max} [{estado_cap}]\n")
        costos_rutas.append(costo_vehiculo)
        costo_total_solucion += costo_vehiculo
        
    reporte.append("="*80)
    reporte.append(f"COSTO TOTAL DE LA SOLUCIÓN: {costo_total_solucion}")
    reporte.append("="*80 + "\n")
    
    # Imprimimos y también retornamos el texto para poder guardarlo
    texto_final = "\n".join(reporte)

    if carpeta_salida:
       guardar_objeto_automatico(carpeta_salida, nombre_instancia, texto_final, "initial_random_solution_detail")

    return costos_rutas, costo_total_solucion, texto_final

In [69]:
c_, cts_, tf = detalle_sol(random_sol_t, d_noreq, G_test, carpeta_salida=tst_path, nombre_instancia=d_noreq['NOMBRE'])

In [70]:
c_

[1835, 1483, 1667, 2935, 2434]

In [63]:
cts_

10354

In [71]:
print(tf)

EVALUACIÓN COMPACTA DE RUTAS
RUTA 1 ['TR10', 'TR21', 'TR41', 'TR26', 'TR37', 'TR19', 'TR34']
  -> TR10 (15,17) -> DH: [1 -> 2 -> 4 -> 5 -> 7 -> 8 -> 9 -> 10 -> 11 -> 12 -> 76 -> 77 -> 15] | Demanda: 26 | Costo (DH + Serv): 274 + 26 = 300
  -> TR21 (32,35) -> DH: [17 -> 15 -> 18 -> 19 -> 21 -> 22 -> 75 -> 23 -> 31 -> 32] | Demanda: 86 | Costo (DH + Serv): 299 + 86 = 385
  -> TR41 (62,63) -> DH: [35 -> 41 -> 42 -> 56 -> 67 -> 62] | Demanda: 30 | Costo (DH + Serv): 126 + 30 = 156
  -> TR26 (46,47) -> DH: [63 -> 62 -> 60 -> 58 -> 59 -> 44 -> 46] | Demanda: 33 | Costo (DH + Serv): 152 + 33 = 185
  -> TR37 (11,59) -> DH: [47 -> 48 -> 11] | Demanda: 78 | Costo (DH + Serv): 89 + 78 = 167
  -> TR19 (32,33) -> DH: [59 -> 58 -> 57 -> 42 -> 41 -> 35 -> 32] | Demanda: 30 | Costo (DH + Serv): 236 + 30 = 266
  -> TR34 (42,57) -> DH: [33 -> 37 -> 39 -> 40 -> 41 -> 42] | Demanda: 14 | Costo (DH + Serv): 128 + 14 = 142
  -> REGRESO A DEPÓSITO (1) -> DH: [57 -> 58 -> 69 -> 4 -> 2 -> 1] | Costo Regreso: 2

In [102]:
random_sol_t[2]

['TR38',
 'TR11',
 'TR9',
 'TR47',
 'TR50',
 'TR28',
 'TR31',
 'TR14',
 'TR6',
 'TR30',
 'TR24',
 'TR29',
 'TR45']

In [106]:
def k_swap(ruta, k):
    hijo = ruta.copy()
    k_real = min(k, len(hijo))
    if k_real < 2: return hijo
        
    indices = random.sample(range(len(hijo)), k_real)
    valores = [hijo[i] for i in indices]
    
    for i in range(k_real):
        hijo[indices[i]] = valores[(i+1) % k_real]
        
    return hijo, indices

In [107]:
random_sol_t[2]

['TR38',
 'TR11',
 'TR9',
 'TR47',
 'TR50',
 'TR28',
 'TR31',
 'TR14',
 'TR6',
 'TR30',
 'TR24',
 'TR29',
 'TR45']

In [108]:
k_swap(random_sol_t[2], 3)

(['TR38',
  'TR30',
  'TR9',
  'TR47',
  'TR50',
  'TR11',
  'TR31',
  'TR14',
  'TR6',
  'TR28',
  'TR24',
  'TR29',
  'TR45'],
 [9, 5, 1])

In [122]:
def k_insertion(ruta, k):
    ruta_ins = ruta.copy()
    k_real = min(k, len(ruta_ins))
    if k_real == 0: return ruta_ins
    
    indices = sorted(random.sample(range(len(ruta_ins)), k_real))
    bloque = [ruta_ins[i] for i in indices]
    
    for i in reversed(indices):
        del ruta_ins[i]
        
    pos = random.randint(0, len(ruta_ins))
    for v in reversed(bloque):
        ruta_ins.insert(pos, v)
        
    return ruta_ins, indices, bloque, pos

In [123]:
random_sol_t[2]

['TR38',
 'TR11',
 'TR9',
 'TR47',
 'TR50',
 'TR28',
 'TR31',
 'TR14',
 'TR6',
 'TR30',
 'TR24',
 'TR29',
 'TR45']

In [124]:
k_insertion(random_sol_t[2], 2)

(['TR38',
  'TR11',
  'TR9',
  'TR47',
  'TR6',
  'TR29',
  'TR50',
  'TR28',
  'TR31',
  'TR14',
  'TR30',
  'TR24',
  'TR45'],
 [8, 11],
 ['TR6', 'TR29'],
 4)

In [157]:
# ==========================================
# OPERADORES EVOLUTIVOS (INTRA E INTER)
# ==========================================

def k_swap(ruta, k):
    ruta_kswap = ruta.copy()
    k_real = min(k, len(ruta_kswap))
    if k_real < 2: return ruta_kswap, {'info': 'Sin cambios (ruta muy corta)'}
        
    indices = random.sample(range(len(ruta_kswap)), k_real)
    valores = [ruta_kswap[i] for i in indices]
    
    for i in range(k_real):
        ruta_kswap[indices[i]] = valores[(i+1) % k_real]
    
    metadata = {
        'indices_afectados': indices,
        'valores_rotados': valores
    }
    return ruta_kswap, metadata

def k_insertion(ruta, k):
    ruta_kins = ruta.copy()
    k_real = min(k, len(ruta_kins))
    if k_real == 0: return ruta_kins, {'info': 'Sin cambios (k=0)'}
    
    indices = sorted(random.sample(range(len(ruta_kins)), k_real))
    bloque = [ruta_kins[i] for i in indices]
    
    for i in reversed(indices):
        del ruta_kins[i]
        
    pos = random.randint(0, len(ruta_kins))
    for v in reversed(bloque):
        ruta_kins.insert(pos, v)
        
    metadata = {
        'indices_origen': indices,
        'bloque_extraido': bloque,
        'posicion_destino': pos
    }
    return ruta_kins, metadata

def k_inversion(ruta, k):
    ruta_kinv = ruta.copy()
    k_real = min(k, len(ruta_kinv))
    if k_real < 2: return ruta_kinv, {'info': 'Sin cambios (ruta muy corta)'}
    
    inicio = random.randint(0, len(ruta_kinv) - k_real)
    fin = inicio + k_real
    
    segmento_original = ruta_kinv[inicio:fin].copy()
    ruta_kinv[inicio:fin] = list(reversed(ruta_kinv[inicio:fin]))
    
    metadata = {
        'inicio': inicio,
        'fin': fin,
        'segmento_original': segmento_original
    }
    return ruta_kinv, metadata

def k_scramble(ruta, k):
    ruta_kscr = ruta.copy()
    k_real = min(k, len(ruta_kscr))
    if k_real < 2: return ruta_kscr, {'info': 'Sin cambios (ruta muy corta)'}
    
    inicio = random.randint(0, len(ruta_kscr) - k_real)
    fin = inicio + k_real
    
    segmento = ruta_kscr[inicio:fin]
    random.shuffle(segmento)
    ruta_kscr[inicio:fin] = segmento
    
    metadata = {
        'inicio': inicio,
        'fin': fin,
        'nuevo_orden': segmento
    }
    return ruta_kscr, metadata

def k_point_crossover(r1, r2, k):
    h1, h2 = r1.copy(), r2.copy()
    k1, k2 = min(k, len(h1)), min(k, len(h2))
    
    start1 = random.randint(0, len(h1) - k1) if len(h1) >= k1 else 0
    start2 = random.randint(0, len(h2) - k2) if len(h2) >= k2 else 0
    
    seg1 = h1[start1:start1+k1]
    seg2 = h2[start2:start2+k2]
    
    h1[start1:start1+k1] = seg2
    h2[start2:start2+k2] = seg1
    
    metadata = {
        'start_r1': start1, 'len_r1': k1, 'seg_r1': seg1,
        'start_r2': start2, 'len_r2': k2, 'seg_r2': seg2
    }
    return h1, h2, metadata

def k_position_crossover(r1, r2, k):
    h1, h2 = r1.copy(), r2.copy()
    k1, k2 = min(k, len(h1)), min(k, len(h2))
    
    idx1 = sorted(random.sample(range(len(h1)), k1), reverse=True)
    idx2 = sorted(random.sample(range(len(h2)), k2), reverse=True)
    
    popped1 = [h1.pop(i) for i in idx1]
    popped2 = [h2.pop(i) for i in idx2]
    
    h1.extend(popped2)
    h2.extend(popped1)
    
    metadata = {
        'idx_r1': idx1, 'popped_r1': popped1,
        'idx_r2': idx2, 'popped_r2': popped2
    }
    return h1, h2, metadata

def aplicar_operador(tipo, padres, k):
    """Despachador central. Captura la ruta y la metadata para el log."""
    if tipo == "mutacion":
        operadores = [k_swap, k_inversion, k_scramble, k_insertion]
        op = random.choice(operadores)
        
        hijos = []
        meta_detalles = []
        for p in padres:
            # Ahora desempaquetamos explícitamente la ruta y el diccionario
            ruta_mutada, meta = op(p, k)
            hijos.append(ruta_mutada) 
            meta_detalles.append(meta) 
            
        return hijos, op.__name__, meta_detalles
        
    elif tipo == "cruce":
        operadores = [k_point_crossover, k_position_crossover]
        op = random.choice(operadores)
        
        hijos = []
        meta_detalles = []
        for i in range(0, len(padres)-1, 2):
            p1 = padres[i]
            p2 = padres[i+1]
            h1, h2, meta = op(p1, p2, k)
            hijos.extend([h1, h2])
            meta_detalles.append(meta)
            
        return hijos, op.__name__, meta_detalles
    else:
        raise ValueError("tipo debe ser 'mutacion' o 'cruce'")

In [166]:
def aplicar_y_evaluar_vecindario(solucion, data, G, matriz_distancias, p_inter=0.5, max_intentos=100, k=2, generar_reporte=True):
    """
    Genera un vecino mediante mutación o cruce.
    Si generar_reporte=False, omite todo el procesamiento de texto para maximizar el rendimiento en CPU.
    """
    intentos = 0
    vecino_encontrado = False
    nueva = []
    detalles_cambio = ""
    
    while intentos < max_intentos:
        intentos += 1
        nueva = copy.deepcopy(solucion)
        
        rutas_con_tareas = [i for i, r in enumerate(nueva) if len(r) > 0]
        
        if not rutas_con_tareas:
            if generar_reporte: detalles_cambio = "No hay tareas en la solución para mover."
            break
            
        es_inter = (random.random() < p_inter)
        
        if es_inter:
            tipo = "cruce"
            r1 = random.choice(rutas_con_tareas)
            
            rutas_disponibles = [i for i in range(len(nueva)) if i != r1]
            if not rutas_disponibles: continue 
                
            r2 = random.choice(rutas_disponibles)
            
            padres = [nueva[r1], nueva[r2]]
            hijos, nombre_op, metadata = aplicar_operador(tipo, padres, k)
            
            nueva[r1] = hijos[0]
            nueva[r2] = hijos[1]
            
            if generar_reporte:
                estado_r2 = "VACIADA" if len(hijos[1]) == 0 else "ACTIVA"
                detalles_cambio = f"CRUCE ({nombre_op}) - Inter-ruta:\n  -> Padres: Ruta {r1+1} y Ruta {r2+1}\n  -> Se intercambiaron k={k} segmentos/elementos. Estado Ruta {r2+1}: {estado_r2}\n  -> Metadata del cruce: {metadata[0]}"
            
        else:
            tipo = "mutacion"
            r1 = random.choice(rutas_con_tareas)
            
            padres = [nueva[r1]]
            hijos, nombre_op, metadata = aplicar_operador(tipo, padres, k)
            
            nueva[r1] = hijos[0]
            
            if generar_reporte:
                detalles_cambio = f"MUTACIÓN ({nombre_op}) - Intra-ruta:\n  -> Operación en Ruta {r1+1} con k={k}\n  -> Metadata de mutación: {metadata[0]}"
            
        # --- VERIFICACIÓN DE FACTIBILIDAD ---
        if es_solucion_factible(nueva, data, matriz_distancias):
            vecino_encontrado = True
            if generar_reporte:
                detalles_cambio += f"\n  -> ¡Movimiento factible encontrado en el intento {intentos}!"
            break 

    # ==========================================
    # SALIDA RÁPIDA (MODO METAHEURÍSTICA)
    # ==========================================
    if not generar_reporte:
        if not vecino_encontrado:
            return solucion, "" # Si falla, retorna la original intacta
        return nueva, ""

    # ==========================================
    # REPORTE DETALLADO (MODO DEBUG/INTERFAZ)
    # ==========================================
    reporte_debug = "\n" + "*"*80 + "\n🔍 DEBUG: APLICADOR GENÉTICO DE VECINDARIO\n" + "*"*80 + "\nSOLUCIÓN ORIGINAL:\n"
    for i, r in enumerate(solucion): reporte_debug += f"  Ruta {i+1}: {r}\n"
    
    if not vecino_encontrado:
        reporte_debug += f"\nOPERACIÓN FALLIDA:\n  -> Tras {max_intentos} intentos, no se pudo mutar/cruzar hacia una solución factible.\n" + "*"*80 + "\n"
        return solucion, reporte_debug
        
    reporte_debug += f"\nDETALLE DEL CAMBIO:\n{detalles_cambio}\n\nVECINO RESULTANTE:\n"
    for i, r in enumerate(nueva): reporte_debug += f"  Ruta {i+1}: {r}\n"
    reporte_debug += "*"*80 + "\n"
    
    return nueva, reporte_debug

In [167]:
new_sol, report_neigh = aplicar_y_evaluar_vecindario(random_sol_t, d_noreq, G_test, matriz_distancias_t, p_inter=0.5, max_intentos=100, k=2, generar_reporte=False)

In [169]:
report_neigh

''

In [168]:
new_sol

[['TR10', 'TR21', 'TR41', 'TR26', 'TR37', 'TR19', 'TR34'],
 ['TR1', 'TR51', 'TR17', 'TR23', 'TR4', 'TR25', 'TR7', 'TR36'],
 ['TR38',
  'TR11',
  'TR9',
  'TR47',
  'TR50',
  'TR28',
  'TR31',
  'TR14',
  'TR6',
  'TR30',
  'TR24',
  'TR29',
  'TR45'],
 ['TR39',
  'TR15',
  'TR49',
  'TR33',
  'TR44',
  'TR13',
  'TR12',
  'TR20',
  'TR16',
  'TR22',
  'TR8',
  'TR43'],
 ['TR2',
  'TR27',
  'TR35',
  'TR5',
  'TR18',
  'TR46',
  'TR48',
  'TR42',
  'TR3',
  'TR32',
  'TR40']]

In [170]:
es_solucion_factible(new_sol, d_noreq, matriz_distancias_t)

True

In [171]:
es_solucion_factible(random_sol_t, d_noreq, matriz_distancias_t)

True

In [172]:
import math

# ==========================================
# 5. METAHEURÍSTICAS (ALTA VELOCIDAD)
# ==========================================

def calcular_costo_rapido(solucion, data, matriz_distancias):
    """Calcula el costo total de una solución en O(1) usando la matriz precalculada."""
    deposito = data.get('DEPOSITO', 1)
    info_tareas = {t['tarea']: t for t in data.get('LISTA_ARISTAS_REQ', [])}
    costo_total = 0
    
    for ruta in solucion:
        if not ruta: continue
        nodo_actual = deposito
        for id_tarea in ruta:
            tarea = info_tareas[id_tarea]
            u, v = tarea['nodos']
            
            costo_total += matriz_distancias[nodo_actual][u] + tarea['costo']
            nodo_actual = v
        costo_total += matriz_distancias[nodo_actual][deposito]
        
    return costo_total

In [173]:
calcular_costo_rapido(new_sol, d_noreq, matriz_distancias_t)

10111

In [174]:
calcular_costo_rapido(random_sol_t, d_noreq, matriz_distancias_t)

10354

In [184]:
## BKS 
import pandas as pd

In [187]:
benchmark = pd.read_csv(r'C:\Users\alhel\Desktop/TESIS CARP 2025/Instancias/Benchmarks.csv')
benchmark

,Instances,BKS,BLB,BUB
0,ksh1,"14,661","14,661","14,661"
1,ksh2,"9,863","9,863","9,863"
2,ksh3,"9,320","9,320","9,320"
3,ksh4,"11,498","11,498","11,498"
4,ksh5,"10,957","10,957","10,957"
...,...,...,...,...
82,egl-s3-b,NaN,"13,648","13,682"
83,egl-s3-c,"17,188","17,188","17,188"
84,egl-s4-a,NaN,"12,153","12,268"
85,egl-s4-b,NaN,"16,113","16,283"


In [188]:
import pandas as pd
import numpy as np

def crear_diccionario_benchmarks(df_benchmarks):
    """
    Convierte un DataFrame de benchmarks en un diccionario de consulta rápida O(1).
    Estructura resultante: {'nombre_instancia': {'BKS': 100.0, 'BLB': 95.0, 'BUB': 105.0}}
    """
    diccionario_bks = {}
    
    if df_benchmarks is None or df_benchmarks.empty:
        return diccionario_bks
        
    # Función interna de limpieza de números
    def limpiar_numero(val):
        if pd.isna(val):
            return None
        try:
            if isinstance(val, str):
                val = val.replace(',', '').strip()
            return float(val)
        except (ValueError, TypeError):
            return None

    # Iteramos sobre el DataFrame y poblamos el diccionario
    for _, row in df_benchmarks.iterrows():
        # Estandarizamos el nombre de la llave (minúsculas y sin espacios a los lados)
        if 'Instances' not in row:
            continue
            
        nombre_instancia = str(row['Instances']).strip().lower()
        
        diccionario_bks[nombre_instancia] = {
            'BKS': limpiar_numero(row.get('BKS')),
            'BLB': limpiar_numero(row.get('BLB')),
            'BUB': limpiar_numero(row.get('BUB'))
        }
        
    return diccionario_bks

In [189]:
benchmark_dict = crear_diccionario_benchmarks(benchmark)
benchmark_dict

{'ksh1': {'BKS': 14661.0, 'BLB': 14661.0, 'BUB': 14661.0},
 'ksh2': {'BKS': 9863.0, 'BLB': 9863.0, 'BUB': 9863.0},
 'ksh3': {'BKS': 9320.0, 'BLB': 9320.0, 'BUB': 9320.0},
 'ksh4': {'BKS': 11498.0, 'BLB': 11498.0, 'BUB': 11498.0},
 'ksh5': {'BKS': 10957.0, 'BLB': 10957.0, 'BUB': 10957.0},
 'ksh6': {'BKS': 10197.0, 'BLB': 10197.0, 'BUB': 10197.0},
 'gdb1': {'BKS': 316.0, 'BLB': 316.0, 'BUB': 316.0},
 'gdb2': {'BKS': 339.0, 'BLB': 339.0, 'BUB': 339.0},
 'gdb3': {'BKS': 275.0, 'BLB': 275.0, 'BUB': 275.0},
 'gdb4': {'BKS': 287.0, 'BLB': 287.0, 'BUB': 287.0},
 'gdb5': {'BKS': 377.0, 'BLB': 377.0, 'BUB': 377.0},
 'gdb6': {'BKS': 298.0, 'BLB': 298.0, 'BUB': 298.0},
 'gdb7': {'BKS': 325.0, 'BLB': 325.0, 'BUB': 325.0},
 'gdb8': {'BKS': 348.0, 'BLB': 348.0, 'BUB': 348.0},
 'gdb9': {'BKS': 303.0, 'BLB': 303.0, 'BUB': 303.0},
 'gdb10': {'BKS': 275.0, 'BLB': 275.0, 'BUB': 275.0},
 'gdb11': {'BKS': 395.0, 'BLB': 395.0, 'BUB': 395.0},
 'gdb12': {'BKS': 458.0, 'BLB': 458.0, 'BUB': 458.0},
 'gdb13': {'B

In [195]:
import pandas as pd

def evaluar_gap_benchmarks(nombre_instancia, costo_actual, diccionario_bks):
    """
    Busca la instancia en el diccionario global y calcula los GAPs disponibles.
    Retorna un DataFrame de Pandas con el resumen de la métrica.
    """
    # Inicializamos valores nulos por defecto en caso de que falten datos
    bks, blb, bub = None, None, None
    gap_bks, gap_blb, gap_bub = None, None, None
    
    # Si tenemos los datos mínimos para buscar
    if diccionario_bks and nombre_instancia:
        nombre_limpio = str(nombre_instancia).strip().lower()
        limites = diccionario_bks.get(nombre_limpio)
        
        if limites:
            bks = limites.get('BKS')
            blb = limites.get('BLB')
            bub = limites.get('BUB')
            
            # Función lambda segura para calcular el porcentaje
            calc_pct = lambda base: ((costo_actual - base) / base) * 100 if base and base > 0 and costo_actual is not None else None

            if bks is not None: gap_bks = calc_pct(bks)
            if blb is not None: gap_blb = calc_pct(blb)
            if bub is not None: gap_bub = calc_pct(bub)

    # Creamos el diccionario estructurado para Pandas
    data = {
        "nombre_instancia": [nombre_instancia],
        "bks": [bks],
        "blb": [blb],
        "bub": [bub],
        "costo_solucion": [costo_actual],
        "gap%": [gap_bks],
        "gap_blb%": [gap_blb],
        "gap_bub%": [gap_bub]
    }
    
    # Retornamos directamente el DataFrame
    return pd.DataFrame(data)

In [196]:
evaluar_gap_benchmarks(d_noreq['NOMBRE'], 10354, benchmark_dict)

,nombre_instancia,bks,blb,bub,costo_solucion,gap%,gap_blb%,gap_bub%
0,egl-e1-A,3548.0,3548.0,3548.0,10354,191.826381,191.826381,191.826381


In [197]:
### RUN FUNCTION

In [176]:
############ SA

In [213]:
import time
import math
import copy
import random
import pandas as pd
# Asegúrate de importar tu módulo core si lo tienes separado, ej: import carp_core as carp

def sa_optimizar(nombre_instancia, solucion_inicial, data, G, matriz_distancias, dict_bks, t_inicial, alfa, iter_por_t, t_final, p_inter=0.5, k=2):
    """
    Ejecuta el Algoritmo de Recocido Simulado (Simulated Annealing).
    Integrado con el motor de vecindarios genéticos y exportación de experimentos.
    Retorna: mejor_solucion, mejor_costo, historial_costos, stats, df_experimento
    """
    # Iniciamos el cronómetro de CPU
    t_inicio_cpu = time.process_time()

    # 1. Inicialización
    solucion_actual = copy.deepcopy(solucion_inicial)
    
    # Asumo que calcular_costo_rapido y aplicar_y_evaluar_vecindario están en el scope 
    # (o prefijadas con tu módulo carp. si vienen de carp_core.py)
    costo_inicial = calcular_costo_rapido(solucion_actual, data, matriz_distancias)
    costo_actual = costo_inicial
    
    mejor_solucion = copy.deepcopy(solucion_actual)
    mejor_costo = costo_actual
    
    T = t_inicial
    historial_costos = []
    
    stats = {
        "p_inter_usado": p_inter,
        "k_usado": k,
        "iteraciones_totales": 0,
        "vecinos_factibles": 0,
        "movimientos_aceptados": 0,
        "mejoras_globales": 0,
        "tiempo_cpu_segundos": 0.0 
    }
    
    # 2. Bucle de Enfriamiento
    while T > t_final:
        for _ in range(iter_por_t):
            stats["iteraciones_totales"] += 1
            
            # Generador híbrido rápido
            vecino, _ = aplicar_y_evaluar_vecindario(
                solucion_actual, data, G, matriz_distancias, 
                p_inter=p_inter, max_intentos=100, k=k, generar_reporte=False
            )
            
            # Si falló la factibilidad
            if vecino == solucion_actual: 
                continue 
                
            stats["vecinos_factibles"] += 1
            costo_vecino = calcular_costo_rapido(vecino, data, matriz_distancias)
            delta = costo_vecino - costo_actual
            
            # 3. Criterio de Aceptación (Metrópolis)
            if delta < 0 or random.random() < math.exp(-delta / T):
                solucion_actual = copy.deepcopy(vecino)
                costo_actual = costo_vecino
                stats["movimientos_aceptados"] += 1
                
                # Actualizar el récord global
                if costo_actual < mejor_costo:
                    mejor_solucion = copy.deepcopy(solucion_actual)
                    mejor_costo = costo_actual
                    stats["mejoras_globales"] += 1
                    
        historial_costos.append(mejor_costo)
        T *= alfa
        
    # Detenemos el cronómetro
    t_fin_cpu = time.process_time()
    stats["tiempo_cpu_segundos"] = t_fin_cpu - t_inicio_cpu
    
    # ==========================================
    # 4. CONSTRUCCIÓN DEL DATAFRAME DEL EXPERIMENTO
    # ==========================================
    # Llamamos a tu función de gaps (asegúrate de llamarla correctamente según tu módulo)
    df_experimento = evaluar_gap_benchmarks(nombre_instancia, mejor_costo, dict_bks)
    
    # Inyectamos los datos del experimento directamente en este DataFrame
    df_experimento.insert(0, "Metaheuristica", "Recocido Simulado (SA)")
    
    # Métricas de Costo y Solución
    df_experimento["Costo_Inicial"] = costo_inicial
    # Convertimos la solución a string para que Pandas no sufra al exportar listas anidadas a CSV
    df_experimento["Mejor_Solucion"] = str(mejor_solucion) 
    
    # Parámetros utilizados
    df_experimento["T_inicial"] = t_inicial
    df_experimento["Alfa"] = alfa
    df_experimento["Iter_por_T"] = iter_por_t
    df_experimento["T_final"] = t_final
    df_experimento["p_inter"] = p_inter
    df_experimento["k"] = k
    
    # Rendimiento Computacional
    df_experimento["Tiempo_CPU_seg"] = stats["tiempo_cpu_segundos"]
    df_experimento["Iteraciones_Totales"] = stats["iteraciones_totales"]
    df_experimento["Vecinos_Factibles"] = stats["vecinos_factibles"]
    df_experimento["Movimientos_Aceptados"] = stats["movimientos_aceptados"]
    df_experimento["Mejoras_Globales"] = stats["mejoras_globales"]
        
    return df_experimento

In [207]:
sa_optimizar(d_noreq['NOMBRE'], random_sol_t, d_noreq, G_test, matriz_distancias_t, benchmark_dict, 1000, 0.9, 1500 , 0.01, p_inter=0.7, k=2)

,Metaheuristica,nombre_instancia,bks,blb,bub,costo_solucion,gap%,gap_blb%,gap_bub%,Costo_Inicial,...,Alfa,Iter_por_T,T_final,p_inter,k,Tiempo_CPU_seg,Iteraciones_Totales,Vecinos_Factibles,Movimientos_Aceptados,Mejoras_Globales
0,Recocido Simulado (SA),egl-e1-A,3548.0,3548.0,3548.0,4367,23.083427,23.083427,23.083427,10354,...,0.9,1500,0.01,0.7,2,89.625,165000,150327,27652,114


In [208]:
##### loop

In [ ]:
def basic_run(instance_path, run_id, benchmark_dict, algoritmo_dist='dijkstra', t_inicial=5000, alfa=0.9, iter_por_t=500, t_final=0.01, p_inter=0.7, k=2):

    ### read_instance
    instance_d, instance_save_name = leer_carplib_dat(instance_path)
    
    ## validate instance reading 
    val_d, val_d_name = validate_instance(instance_d, instance_path)
    
    if not val_d['is_valid']:
        return "reading instance failed"

    ## run folder
    instance_name = instance_d['NOMBRE']
    # Manteniendo tu instrucción de ruta: nombre_instancia/id
    base_dir_run = f"{instance_name}/{run_id}"
    
    path_saving = iniciar_ejecucion(instance_d, base_dir=base_dir_run)
    
    ## grafo
    g_object = generar_grafo_limpio(instance_d, carpeta_salida=path_saving, figsize=(35,30), k_layout=15)
    
    ### distance matrix
    matriz_caminos = calcular_matriz_distancias(g_object, algoritmo=algoritmo_dist, carpeta_salida=path_saving, nombre_instancia=instance_name)

    #### random_init_sol
    random_init_sol = generar_solucion_inicial_aleatoria(instance_d, matriz_caminos, max_intentos=1000, carpeta_salida=path_saving, nombre_instancia=instance_name)

    #### start SA
    mejor_solucion, mejor_costo, historial_costos, stats, df_experimento = sa_optimizar(
        instance_name, random_init_sol, instance_d, g_object, matriz_caminos,
        benchmark_dict, t_inicial, alfa, iter_por_t, t_final, p_inter, k
    )
    
    # ==========================================
    # INYECCIÓN DE METADATOS AL DATAFRAME
    # ==========================================
    # Insertamos las columnas al principio (índices 1 y 2) para tener el contexto claro
    df_experimento.insert(1, "Run_ID", run_id)
    df_experimento.insert(2, "Algoritmo_Distancias", algoritmo_dist)
    
    return df_experimento

In [228]:
def basic_run(instance_path, run_id, benchmark_dict, algoritmo_dist='dijkstra', t_inicial=5000, alfa=0.9, iter_por_t=500, 
              t_final=0.01, p_inter=0.7, k=2):

    ### read_instance
    instance_d, instance_save_name = leer_carplib_dat(instance_path)
    
    ## validate instance reading 
    val_d, val_d_name = validate_instance(instance_d, instance_path)
    
    # Usamos la lógica booleana invertida y un return temprano
    if not val_d['is_valid']:
        return "reading instance failed"

    ## run folder
    instance_name = instance_d['NOMBRE']
    # Formato de ruta: nombre_instancia/id
    base_dir_run = f"{instance_name}/{run_id}"
    
    # Corregido: pasamos instance_d en lugar de d_noreq
    path_saving, logger = iniciar_ejecucion(instance_d, base_dir=base_dir_run)

    ## grafo
    g_object = generar_grafo_limpio(instance_d, carpeta_salida=path_saving, figsize=(35,30), k_layout=15)
    
    ### distance matrix
    matriz_caminos = calcular_matriz_distancias(g_object, algoritmo=algoritmo_dist, carpeta_salida=path_saving, nombre_instancia=instance_name)

    #### random_init_sol
    random_init_sol = generar_solucion_inicial_aleatoria(instance_d, matriz_caminos, max_intentos=1000, carpeta_salida=path_saving,
                                                         nombre_instancia=instance_name)

    #### start SA
    # Desempaquetamos los 5 valores y nos quedamos con el DataFrame
    df_experimento = sa_optimizar(
        instance_name, random_init_sol, instance_d, g_object, matriz_caminos,
        benchmark_dict, t_inicial, alfa, iter_por_t, t_final, p_inter, k
    )
    
    # Opcional pero muy útil: Inyectar el run_id al dataframe para rastrear la corrida
    df_experimento.insert(1, "Run_ID", run_id)
    df_experimento.insert(2, "Algoritmo_Distancias", algoritmo_dist)

    ruta_archivo = f"{path_saving}/resultados.csv"
    df_experimento.to_csv(ruta_archivo, index=False)
    
    return df_experimento

In [229]:
basic_run(ruta_archivo_norequ, '000_TESTING_SA', benchmark_dict, algoritmo_dist='dijkstra', t_inicial=5000, alfa=0.9, iter_por_t=500, 
              t_final=0.01, p_inter=0.7, k=2)

,Metaheuristica,Run_ID,Algoritmo_Distancias,nombre_instancia,bks,blb,bub,costo_solucion,gap%,gap_blb%,...,Alfa,Iter_por_T,T_final,p_inter,k,Tiempo_CPU_seg,Iteraciones_Totales,Vecinos_Factibles,Movimientos_Aceptados,Mejoras_Globales
0,Recocido Simulado (SA),000_TESTING_SA,dijkstra,egl-e1-A,3548.0,3548.0,3548.0,4768,34.385569,34.385569,...,0.9,500,0.01,0.7,2,32.9375,62500,56948,16213,100
